# **Setting Up a Project Folder** on Google Drive with R and Google Colab

One of the most common sources of confusion in data analysis is **not knowing where your files live**. This lecture establishes a clean, reproducible folder structure you can use for every project.

By the end of this notebook you will be able to:

1. Create a project folder in Google Drive with subfolders for scripts, raw data, clean data, and figures.
2. Mount Google Drive inside Google Colab so your R code can read and write files there.
3. Follow a two-script workflow: A) `1_data_clean` reads from `raw_data/` and writes to `clean_data/` B) `2_analysis` reads from `clean_data/` and writes to `figures/`.

---

**Important:** This notebook is designed to run in **Google Colab** with an **R runtime**. It will not work in a standard local Jupyter session.

To switch to the R runtime in Colab: `Runtime → Change runtime type → R`.

# Why Organize Your Project This Way?

File structure can make or break a research project

*   Acts as an outline that gives you forward momentum
*   Allows your collaborators to clearly follow what you did
*   Allows you to return to a project after and quickly remember what step you were on and what
remains to be done

There are best practices for file structure that are worth following now

```
my_project/
│
├── raw_data/        ← original files, NEVER modified
├── clean_data/      ← output of 1_data_clean
├── figures/         ← output of 2_analysis
├── scripts/         ← folder for all code scripts, numbered in order they run
├──── 1_data_clean.ipynb
└──── 2_analysis.ipynb
```

This separation means:
- Your raw data is preserved exactly as you received it. You never know what you thought was a good idea may turn out to be a bad one!
- You can always re-run `1_data_clean` from scratch and reproduce your clean dataset.
- You can always re-run `2_analysis` from the clean dataset and reproduce every figure, allowing you to skip cleaning the data every time you go to work on your analysis.


# Step 0 — Mount Google Drive

Google Colab's R runtime does not have a built-in `drive.mount()` function like the Python runtime does. Therefore, we are going to set this up using **python** and then switch back to coding in **R**.

Change the run time to Python, then run the cell below. Colab will ask you to authorize access to your Google Drive.

In [ ]:
# Make sure your runtime is Python for this one block of code.
from google.colab import drive
drive.mount('/content/drive')

# Step 1 — Create the Folder in Google Drive (Done in the Browser)

Before writing any code, go to [drive.google.com](https://drive.google.com) and create a new folder. For this example, name it `national_parks_project`. This is where your whole project will live.

For future projects (*i.e.,* your capstone) you will repeat these steps.

You do **not** need to create the subfolders (`raw_data`, `figures` etc.) manually — the code below will create them for you.

When you already have raw data (a CSV file, for example), you will be able to upload it to Google Drive. For this example, we will move our raw data into the `raw_data/` subfolder after the code creates our dataset.

# Step 2 — Set Your Project Folder as the Working Directory

The **working directory** is the folder R uses as its home base. When you run `read_csv("raw_data/myfile.csv")`, R looks for that file *relative to the working directory*. Setting it once at the top of your script means you never have to type long absolute paths again.

Let's use our `national_parks_project` as our working directory.

Before running the next code block, **change the runtime back to R!!**`

In [ ]:
# ---------------------------------------------------------
# Setting the working directory
# ---------------------------------------------------------
my_project_folder <- "national_parks_project"

# Build the full path
project_path <- file.path("/content/drive/MyDrive", my_project_folder)

# Create the project folder if it does not exist yet
dir.create(project_path, showWarnings = FALSE)

# Set it as the working directory
setwd(project_path)

# Confirm
getwd()

# Step 3 — Create the Subfolder Structure

We create four subfolders. The argument `showWarnings = FALSE` means R will not complain if the folder already exists — useful when re-running the notebook.

In [ ]:
dir.create("raw_data",   showWarnings = FALSE)
dir.create("clean_data", showWarnings = FALSE)
dir.create("figures",    showWarnings = FALSE)
dir.create("scripts",    showWarnings = FALSE)

# Verify
list.files()

You should now see `raw_data`, `clean_data`, `scripts` and `figures` listed. These folders also appear immediately in your Google Drive — open Drive in your browser and navigate into your project folder to confirm.

**Note:** If you had raw data to upload, at this point you would put it in the `raw_data` folder so that it was a part of the project.

Let's now move *this* notebook into the `scripts/` folder. This would let you find this Colab Notebook in your Google Drive more easily, because it will be a part of your `national_parks_project/`.

---

# Script 1 — `1_data_clean`: Read Raw Data, Clean It, Write to `clean_data/`

In a real project this step would be it's own notebook. For leaning purposes, today we will just have it be it's own section.

**The rule:** read from `raw_data/`, write to `clean_data/`. *Never overwrite raw data.*

## A. Read Raw Data

Because we set the working directory to our project folder, the path to read in raw data would just be `"raw_data/filename.csv"` — short and readable.

The cell below creates a small example dataset and saves it to `raw_data/` so the notebook is self-contained. In your own project, you would skip the next code block and instead upload your raw data into the `raw_data/` folder.

In [ ]:
# Load packages
library(readr)   # read_csv / write_csv
library(dplyr)   # data manipulation

# ── Simulate raw data ────────────────────────────────────────────────────
# In a real project, your raw CSV already lives in raw_data/.
# We generate example data here only to make this notebook self-contained.

set.seed(42)
n <- 120

df_raw <- tibble(
  park       = sample(c("Yellowstone", "Yosemite", "Olympic", "Acadia"), n, replace = TRUE),
  species    = sample(c("Black Bear", "Elk", "Coyote", "Bald Eagle", NA), n, replace = TRUE),
  count      = sample(c(1:20, NA), n, replace = TRUE),
  year       = sample(2018:2023, n, replace = TRUE),
  surveyor   = sample(c("A. Creel", "J. Smith", "B. Jones"), n, replace = TRUE)
)

head(df_raw)

# Save to raw_data/ — this is what you would have already uploaded
write_csv(df_raw, "raw_data/species_counts_raw.csv")

cat("Raw file written.\n")

In [ ]:
# ── Read the raw data ─────────────────────────────────────────────────────
# This is the first real line of a data-cleaning script:
# read from raw_data/, do not touch the file after this point.

df_raw <- read_csv("raw_data/species_counts_raw.csv")

head(df_raw)

## 1b. Clean

Now, data cleaning can be a long process. But let's just do a short example of some data cleaning so we can save the "cleaned" data to the `clean_data/` folder.

In [ ]:
# ── Clean the data ────────────────────────────────────────────────────────
df_clean <- df_raw %>%
  filter(!is.na(species), !is.na(count)) %>% # Drop rows where species or count is missing — we cannot use them
  # Rename 'count' to something more descriptive
  rename(n_individuals = count)

cat("Rows in raw data:  ", nrow(df_raw), "\n")
cat("Rows after cleaning:", nrow(df_clean), "\n")
cat("Rows dropped:       ", nrow(df_raw) - nrow(df_clean), "\n")

glimpse(df_clean)

## 1c. Write the Clean Dataset to `clean_data/`

We use `write_csv()` from the `readr` package. The argument `index = FALSE` is not needed in R (unlike Python) — `write_csv()` does not write row numbers by default.

In [ ]:
# Write clean data to clean_data/
write_csv(df_clean, "clean_data/species_counts_clean.csv")

# Verify the file exists
list.files("clean_data/")

Script 1 is done. Check your Google Drive — you should see `clean_data/species_counts_clean.csv` sitting there.

---

# Script 2 — `2_analysis`: Read from `clean_data/`, Write Figures to `figures/`

In a real project, this would be a separate notebook that you and anyone on your team can run, so long as the clean data is already there. It reads from `clean_data/` and never touches `raw_data/`.

This can be nice, because the data cleaning step can often be computationally expensive. So having your analysis in a seperate script means you do not have to run that step again, and can start straight from the cleaned data for the analysis step.

## 2a. Read the Clean Dataset

In [ ]:
# Load packages
library(readr)
library(dplyr)
library(ggplot2)

# Read from clean_data/
df <- read_csv("clean_data/species_counts_clean.csv")

head(df)

## 2b. Make a Figure

In [ ]:
# Count observations per species per park
df_summary <- df %>%
  count(park, species, name = "observations")

# Build the plot
p <- ggplot(df_summary, aes(x = park, y = observations, fill = species)) +
  geom_col(position = "dodge") +
  labs(
    title    = "Wildlife Observations by National Park",
    subtitle = "Cleaned survey data, 2018–2023",
    x        = "Park",
    y        = "Number of observations",
    fill     = "Species"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

p

## 2c. Save the Figure to `figures/`

`ggsave()` saves the last plot by default (or you can pass the plot object explicitly with `plot = p`). Dots per inch, `dpi = 150`, gives a sharp image suitable for a presentation or report.

**For naming your plot or table:** When you have multiple scripts, you should numebr them the order that they should be run on. Ex: `1_data_clean.R`, `2_analysis.R`. When you save figures our tables, the name or those files should start with the same number as the script that created them. For large projects, this lets you know where a figure was made so that you can find that code if you want to edit it (change labels, colors, etc).

In [ ]:
ggsave(
  filename = "figures/2_species_by_park.png",
  plot     = p,
  width    = 8,
  height   = 5,
  dpi      = 150
)

# Verify
list.files("figures/")

---

# What Your Project Folder Looks Like Now

Open Google Drive and navigate into your project folder. You should see:

```
national_parks_project/
│
├── raw_data/
│   └── species_counts_raw.csv      ← original data, never modified
│
├── clean_data/
│   └── species_counts_clean.csv    ← output of Script 1 (1_data_clean)
│
└── figures/
│    └── 2_species_by_park.png      ← output of Script 2 (2_analysis)
│
└── scripts/
    └── google_drive_setup.ipynb     ← Where we have been writing our code.
                                     In a real project, would have 1_data_clean and 2_analysis etc.
```

**Key takeaways:**

- `raw_data/` is read-only. You never write to it from a script.
- `clean_data/` is the handoff point between cleaning and analysis.
- `figures/` holds your final outputs.
- `scripts/` will keep all your Colab notebooks in a place you can easily find them.
- Setting `setwd()` once at the top of each script keeps all paths short and relative.
- Because everything lives on Google Drive, your work is automatically backed up and accessible from any computer.